In [1]:
from pylablib.devices.Thorlabs import MFF # Flip mount
help(MFF)

Help on class MFF in module pylablib.devices.Thorlabs.kinesis:

class MFF(KinesisDevice)
 |  MFF(conn)
 |  
 |  MFF (Motorized Filter Flip Mount) device.
 |  
 |  Implements FTDI chip connectivity via pyft232 (virtual serial interface).
 |  
 |  Args:
 |      conn: serial connection parameters (usually 8-digit device serial number).
 |  
 |  Method resolution order:
 |      MFF
 |      KinesisDevice
 |      pylablib.devices.interface.stage.IMultiaxisStage
 |      pylablib.devices.interface.stage.IStage
 |      BasicKinesisDevice
 |      pylablib.core.devio.comm_backend.ICommBackendWrapper
 |      pylablib.core.devio.interface.IDevice
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, conn)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  get_flipper_parameters(self, channel=None)
 |      Get current flipper parameters ``(transit_time, io1_oper_mode, io1_sig_mode, io1_pulse_width, io2_oper_mode, io2_sig_mode, io2_pulse_width)``


In [1]:
from thorlabs_elliptec import ELLx, ELLError, ELLStatus, list_devices
import thorlabs_elliptec
print(list_devices())
# Prints something like:
# device=/dev/ttyUSB1, manufacturer=Prolific Technology Inc., product=USB-Serial Controller, vid=0x067b, pid=0x2303, serial_number=None, location=1-1.1

device=COM3, manufacturer=FTDI, product=None, vid=0x0403, pid=0x6015, serial_number=DK0IVV6GA, location=None


In [3]:
import nidaqmx

In [4]:
help(nidaqmx)

Help on package nidaqmx:

NAME
    nidaqmx

PACKAGE CONTENTS
    __main__
    _base_interpreter
    _bitfield_utils
    _grpc_interpreter
    _grpc_time
    _install_daqmx
    _lib
    _lib_time
    _library_interpreter
    _linux_installation_commands
    _stubs (package)
    _time
    constants
    error_codes
    errors
    grpc_session_options
    scale
    stream_readers
    stream_writers
    system (package)
    task (package)
    types
    utils

DATA
    __all__ = ['errors', 'scale', 'stream_readers', 'stream_writers', 'tas...

VERSION
    1.0.2

FILE
    c:\users\rn487\anaconda3\lib\site-packages\nidaqmx\__init__.py




In [2]:
import HEDS

In [3]:
help(HEDS.SDK.Init)

Help on function Init in module holoeye_slmdisplaysdk_types:

Init(major, minor)
    ## Initializes SLM Display SDK and checks for the given version numbers to be fulfilled to be able to
    ## detect API incompatibilities between the actually used SDK version and the user code version.
    ## \param major The major version number of SLM Display SDK your code is meant to be run with.
    ## \param minor The minor version number of SLM Display SDK your code is meant to be run with.
    ## \return HEDSERR_NoError when there is no error.Please use \ref HEDS::SDK::ErrorString() to retrieve further error information.



In [4]:
from hedslib.heds_types import *

In [7]:
help(assert)

SyntaxError: invalid syntax (3926837683.py, line 1)

In [2]:
print(thorlabs_elliptec.find_device())

COM3 - USB Serial Port (COM3)


In [5]:
help(ELLx)

Help on class ELLx in module thorlabs_elliptec:

class ELLx(builtins.object)
 |  ELLx(serial_port=None, x: int = None, device_serial: str = None, device_id: int = 0, **kwargs)
 |  
 |  Generic class to interact with the Thorlabs Elliptec series of devices.
 |  
 |  The ``serial_port`` parameter may be a system-specific string (eg. ``"/dev/ttyUSB0"``,
 |  ``"COM12"``) or a :data:`serial.tools.list_ports_common.ListPortInfo` instance. If the
 |  ``serial_port`` parameter is ``None`` (default), then an attempt to detect a serial device will
 |  be performed. The first device found will be initialised. If multiple serial devices are present
 |  on the system, then the use of the the additional keyword arguments can be used to select a
 |  specific device. The keyword arguments the same as those used for :meth:`find_device`.
 |  
 |  The multi-drop feature of the ELLx devices may be used by specifying an existing instance of an
 |  ELLx class as the ``serial_port`` parameter. The serial por

In [3]:
# For an ELL14 set to use device ID 1 on serial port device /dev/ttyUSB0
stage1 = ELLx(x=14, serial_port="COM3", device_id=1)
#stage2 = ELLx(x=14, serial_port="COM3", device_id=2)

In [4]:
print(f"{stage1.model_number} #{stage1.device_id} on {stage1.port_name}, serial number {stage1.serial_number}, status {stage1.status.description}")

ELL14/M #1 on COM3, serial number 11401337, status ok


In [5]:
stage2 = ELLx(x=14, serial_port=stage1, device_id=2)

In [6]:
print(f"{stage2.model_number} #{stage2.device_id} on {stage2.port_name}, serial number {stage2.serial_number}, status {stage2.status.description}")

ELL14/M #2 on COM3, serial number 11401440, status ok


In [17]:
# Move device to the home position
stage1.home()
stage2.home()

In [12]:
# Movements are in real units appropriate for the device (degrees, mm).
stage1.move_absolute(90)
stage2.move_absolute(45.0)

In [19]:
# By default, move commands are asynchronous (non-blocking) and return immediately,
# but you can manually wait for it to be in position
stage1.move_absolute(45.0)
print(stage1.is_moving())
stage1.wait()
print(stage1.is_moving())
print(f"{stage1.get_position()}{stage1.units}")
stage1.move_relative(15)
stage1.wait()
print(f"{stage1.get_position()}{stage1.units}")

True
False
45.003°
60.002°


In [22]:
# Synchronous behaviour can also be achieved by setting the blocking=True parameter,
# which will perform the wait before returning from each movement command.
stage1.home(blocking=True)
print(f"{stage1.get_position()}{stage1.units}")
stage1.move_absolute(1.23, blocking=True)
print(f"{stage1.get_position()}{stage1.units}")
stage1.move_relative(-0.98, blocking=True)
print(f"{stage1.get_position()}{stage1.units}")

0.005°
1.238°
0.256°


In [24]:
# When using the synchronous behaviour, any error during movement will raise an exception.
try:
    stage1.move_absolute(-9999, blocking=True)
except ELLError as ex:
    if ex.status == ELLStatus.OUT_OF_RANGE:
        # Requested move beyond device limits
        print("Device can't move there!")
    else:
        # Other error, eg stage held or blocked so it can't move
        print(f"Movement error: {ex}")
else:
    print("Move completed OK")

Device can't move there!


Device #1 reported status OUT_OF_RANGE (12) out of range = out of range


In [26]:
# When using asynchronous calls, any errors won't have been detected yet,
# so instead, the is_moving() and wait() methods can raise the exception instead.
stage1.move_relative(300)
try:
    print(stage1.is_moving(raise_errors=True))
    stage1.wait(raise_errors=True)
    print(stage1.is_moving(raise_errors=True))
except ELLError as ex:
    print(f"Movement error: {ex}")

True
False


In [ ]:
# or test whether movement is still in progress.
print(stage.is_moving())
stage.wait()
print(stage.is_moving())
print(f"{stage.get_position()}{stage.units}")
# Prints something like:
# True
# False
# 32.655°

# Synchronous behaviour can also be achieved by setting the blocking=True parameter,
# which will perform the wait before returning from each movement command.
stage.home(blocking=True)
stage.move_absolute(1.23, blocking=True)
stage.move_relative(-0.98, blocking=True)

# When using the synchronous behaviour, any error during movement will raise an exception.
try:
    stage.move_absolute(-9999, blocking=True)
except ELLError as ex:
    if ex.status == ELLStatus.OUT_OF_RANGE:
        # Requested move beyond device limits
        print("Device can't move there!")
    else:
        # Other error, eg stage held or blocked so it can't move
        print(f"Movement error: {ex}")
else:
    print("Move completed OK")

# When using asynchronous calls, any errors won't have been detected yet,
# so instead, the is_moving() and wait() methods can raise the exception instead.
stage.move_relative(300)
try:
    print(stage.is_moving(raise_errors=True))
    stage.wait(raise_errors=True)
    print(stage.is_moving(raise_errors=True))
except ELLError as ex:
    print(f"Movement error: {ex}")

# Once done with the device, it can be specifically closed. Commands to the stage will no
# longer work until the device is re-initialised.
stage.close()

In [7]:
stage1.close()
stage2.close()